# 06 时间验证与结果汇总

这本 Notebook 在同一批时间验证患者上比较 ResNet、临床、ResNet 联合临床、影像组学，以及影像组学联合临床五组模型，汇总预测概率、分类指标、ROC 和校准图。

模型、数据处理参数和分类阈值均来自前面的开发阶段。完成 02 的模型保存后即可验证三组分类基线；加入影像组学需要先完成 01，加入自动掩模实验还需要 05 的验证预测。

下面先读取配置，定位本次实验的模型和结果文件。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    settings = path.parent / "experiment_settings.json"
    if settings.is_file():
        overlay = json.loads(settings.read_text(encoding="utf-8"))
        allowed = {"annotation_scope", "optimized", "radiomics", "segmentation", "report_version"}
        if set(overlay) - allowed:
            raise ValueError("experiment_settings.json 只放实验参数，机器路径仍在 config.json。")
        for key, value in overlay.items():
            if isinstance(value, dict):
                cfg[key] = {**cfg.get(key, {}), **value}
            else:
                cfg[key] = value
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg




def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics" / experiment_name(cfg, "radiomics", "selected2d_v1"),
        "explain": root / "explain", "segmentation": root / "segmentation" / experiment_name(cfg, "segmentation", "selected2d_v1"),
        "optimized": root / "optimized" / experiment_name(cfg, "optimized", "resnet_v1"),
        "reports": root / "reports" / safe_name(cfg.get("report_version", "comparison_v2")),
    }




def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")




def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))




def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()




def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result




def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "resnet_baseline_and_optimized_selected_slices", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)




def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table




def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)




def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)




def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    manifest["artifacts"] = {portable_name(k): v for k, v in manifest["artifacts"].items()}
    for key in ("cache", "cohort"):
        manifest[key] = portable_name(manifest[key])
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")


    # Notebook hashes are provenance, not data identity. Model contracts are checked separately.


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本操作需要原分辨率影像，请在数据电脑执行；导入结果和生成报告无需原影像。")




def safe_name(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("实验名称只使用字母、数字、下划线或短横线。")
    return value




def experiment_name(cfg, stage, default):
    return safe_name(cfg.get(stage, {}).get("experiment", default))




def portable_name(value):
    from pathlib import PurePosixPath
    name = str(value).replace("\\", "/")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(p in ("..", ".") or ":" in p for p in path.parts):
        raise ValueError(f"数据包必须使用包内相对路径：{value}")
    return path.as_posix()




def contained_file(root, name):
    root = Path(root).resolve()
    target = (root / portable_name(name)).resolve()
    target.relative_to(root)
    return target




def cohort_identity(cfg):
    """Patient identity independent of local directories, CSV formatting and row order."""
    import pandas as pd
    table = read_cohort(cfg)
    columns = ["mr", "label", "split", "development_fold", "max_layer", "moca"] + CLINICAL
    if not set(columns) <= set(table):
        raise ValueError("病例清单缺少患者、结局、分组、取层或临床列。")
    rows = []
    for _, row in table.sort_values("mr").iterrows():
        values = {}
        for key in columns:
            value = row[key]
            values[key] = (None if pd.isna(value) else str(value) if key in ("mr", "split") else format(float(value), ".12g"))
        rows.append(values)
    payload = {"schema": 1, "patients": rows}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()




def definition_contract(cfg, prefix, names):
    import ast
    files = list(cfg["_root"].glob(prefix + "_*.ipynb"))
    if len(files) != 1:
        raise ValueError(f"找不到唯一的 {prefix} Notebook。")
    notebook = json.loads(files[0].read_text(encoding="utf-8"))
    definitions = {}
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                if node.name in definitions:
                    raise ValueError(f"重复定义：{node.name}")
                definitions[node.name] = hashlib.sha256(ast.dump(node, include_attributes=False).encode()).hexdigest()
    if set(definitions) != set(names):
        raise ValueError("模型或处理流程定义不齐全。")
    return definitions




def verify_transfer(cfg, directory, kind=None):
    directory = Path(directory).resolve()
    manifest = json.loads((directory / "transfer_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema") != 1 or manifest.get("kind") not in ("radiomics_features", "radiomics_complete", "segmentation_results", "segmentation_weights"):
        raise ValueError("不支持的数据包类型或版本。")
    if kind and manifest["kind"] != kind:
        raise ValueError("数据包类型不匹配。")
    if manifest["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("两台机器的患者、标签、临床数据、取层或开发折不一致。")
    if manifest["native_source_identity"] != native_source_identity(cfg):
        raise ValueError("数据包与分类缓存记录的原始影像或人工标注不是同一版本。")
    if manifest["annotation_scope"] != "selected_slices" or cfg["annotation_scope"] != "selected_slices":
        raise ValueError("当前流程只接受四层标注协议。")
    for name, checksum in manifest["artifacts"].items():
        file = contained_file(directory, name)
        if not file.is_file() or digest(file) != checksum:
            raise ValueError(f"数据包文件缺失或传输后发生变化：{name}")
    return manifest




def export_transfer(cfg, source, filenames, destination, kind, protocol):
    import shutil
    source, destination = Path(source).resolve(), Path(destination).resolve()
    fresh(destination)
    mapping = {}
    for name in filenames:
        name = portable_name(name)
        file = contained_file(source, name)
        if not file.is_file():
            raise FileNotFoundError(file)
        mapping[name] = digest(file)
    destination.mkdir(parents=True)
    for name in mapping:
        target = contained_file(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), target)
    dump_json(destination / "transfer_manifest.json", {
        "schema": 1, "kind": kind, "cohort_identity": cohort_identity(cfg), "native_source_identity": native_source_identity(cfg),
        "annotation_scope": cfg["annotation_scope"], "protocol": protocol,
        "artifacts": mapping, "source_run_id": cfg["run_id"],
        "notebook_definitions_sha256": source_snapshot(cfg)})
    verify_transfer(cfg, destination, kind)
    return destination




def import_transfer(cfg, source, stage):
    import shutil
    source = Path(source).resolve()
    manifest = verify_transfer(cfg, source)
    allowed = {"radiomics": ("radiomics_features", "radiomics_complete"),
               "segmentation": ("segmentation_results",), "segmentation_weights": ("segmentation_weights",)}
    if stage not in allowed or manifest["kind"] not in allowed[stage]:
        raise ValueError("数据包与接收阶段不匹配。")
    expected_protocol = "selected_four_slices_2d_pixels_v1"
    if manifest["protocol"] != expected_protocol:
        raise ValueError("数据包的二维取层/像素空间处理协议不匹配。")
    target = paths(cfg)[stage] if stage != "segmentation_weights" else paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    fresh(target)
    target.mkdir(parents=True)
    for name in manifest["artifacts"]:
        dest = contained_file(target, name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), dest)
    shutil.copy2(source / "transfer_manifest.json", target / "transfer_manifest.json")
    verify_transfer(cfg, target)
    return target




def native_source_identity(cfg):
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    patients = read_cohort(cfg).mr.tolist()
    files = IMAGE_FILES + ["Untitled.nii.gz"]
    selected = {mr: {name: audit["patients"][mr]["files"][name]["sha256"] for name in files} for mr in patients}
    return hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()




In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


## 模型推理所需的工具

这里加载 PyTorch、ResNet50 和相关工具，供后面构建网络并恢复训练参数。`DEVICE` 决定模型及影像张量的计算位置，预测函数按照设定的批大小分批处理患者。


In [ ]:
from __future__ import annotations
import copy
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

SEED = 20260922
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SINGLE_SEQUENCES = {"t1": 0, "t2": 1, "dwi": 2}
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 分类网络与概率预测

`MaskEncoder` 和 `DualBranchModel` 定义训练时使用的双分支结构：每层 MRI 与病灶特征拼接，再通过注意力汇总四层信息。`predict_proba` 将模型切换到评估状态，关闭梯度计算，分批返回阳性概率。

这里恢复网络结构后，将加载 02 保存的最终开发模型参数。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: MaskEncoder
class MaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


# 直接复制：code/train_resnet50_manual_cv.py :: DualBranchModel
class DualBranchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


# 直接复制：code/train_resnet50_manual_cv.py :: predict_proba
@torch.no_grad()
def predict_proba(
    model: nn.Module,
    X: torch.Tensor,
    M: torch.Tensor,
    batch_size: int = BATCH_SIZE,
) -> np.ndarray:
    model.eval()
    outs = []
    for i in range(0, len(X), batch_size):
        xb = X[i : i + batch_size].to(DEVICE)
        mb = M[i : i + batch_size].to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outs.append(torch.sigmoid(model(xb, mb)).float().cpu().numpy())
    return np.concatenate(outs)


## 核对模型版本并预测三组分类基线

`verify_lock` 对照开发阶段保存的校验信息，核对模型、阈值、队列、缓存和代码定义。`load_original_model` 随后载入最终 ResNet 权重。

`predict_original_validation` 从患者清单中取出时间验证病例，计算 ResNet 概率，再分别得到临床模型和 ResNet 联合临床模型的概率。分类标签使用各模型在开发阶段确定的阈值，并与 MR 编号、真实标签一起整理成表。

如果传入自动掩模缓存，函数会先核对患者顺序和数组形状，再运行相同分类模型，给结果加上自动掩模替换的标记。


In [ ]:
from __future__ import annotations

def configure(cfg):
    p = paths(cfg)
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    return p







def verify_lock(cfg):
    p = paths(cfg)
    metadata = json.loads((p["temporal"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if digest(p["cache"]) != metadata["cache_sha256"]:
        raise ValueError("影像缓存与开发锁定版本不一致。")
    for name, expected in metadata["artifacts"].items():
        if digest(p["run"] / portable_name(name)) != expected:
            raise ValueError(f"固定产物发生变化：{name}")
    contract = baseline_contract(cfg)
    expected = metadata.get("baseline_contract", ORIGINAL_INFERENCE_CONTRACT)
    if contract != expected:
        raise ValueError("原基线的网络或推理定义发生变化，请使用对应模型版本。")







def load_original_model(cfg, checkpoint=None):
    import torch
    configure(cfg)
    model = DualBranchModel().to(DEVICE)
    checkpoint = checkpoint or paths(cfg)["temporal"] / "final_development_model.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
    model.eval()
    return model







def predict_original_validation(cfg, automatic_masks=None):
    import joblib
    import torch
    verify_lock(cfg)
    p = configure(cfg)
    val = read_cohort(cfg).query("split == 'validation'").reset_index(drop=True)
    idx = val.cache_index.to_numpy(dtype=int)
    with np.load(p["cache"], allow_pickle=False) as cache:
        X = torch.from_numpy(cache["images"][idx].astype(np.float32))
        M = torch.from_numpy(cache["masks"][idx].astype(np.uint8))
    suffix = ""
    if automatic_masks is not None:
        with np.load(automatic_masks, allow_pickle=False) as predicted:
            if predicted["mrs"].astype(str).tolist() != val.mr.tolist() or predicted["masks"].shape != tuple(M.shape):
                raise ValueError("自动掩模患者顺序或形状不匹配。")
            M = torch.from_numpy(predicted["masks"].astype(np.uint8))
        suffix = "_AutoMask_Substitution"
    probability = predict_proba(load_original_model(cfg), X, M)
    dl_metrics = json.loads((p["temporal"] / "development_cv_metrics.json").read_text(encoding="utf-8"))
    meta = json.loads((p["temporal"] / "clinical_fusion_meta.json").read_text(encoding="utf-8"))
    models = joblib.load(p["temporal"] / "clinical_fusion.joblib")
    clinical = val[CLINICAL].to_numpy(dtype=np.float32)
    probabilities = {"ResNet50" + suffix: (probability, dl_metrics["youden_threshold"]),
        "ResNet50_Clinical5" + suffix: (models["ResNet50_Clinical5"].predict_proba(np.column_stack([probability, clinical]))[:, 1], meta["ResNet50_Clinical5"]["threshold"])}
    if not suffix:
        probabilities["Clinical5"] = (models["Clinical5"].predict_proba(clinical)[:, 1], meta["Clinical5"]["threshold"])
    predictions, summaries = [], []
    for name, (prob, threshold) in probabilities.items():
        table = val[["mr", "label"]].copy()
        table["model"], table["probability"], table["threshold"] = name, prob, threshold
        table["prediction"] = (prob >= threshold).astype(int)
        table["protocol"] = "manual_model_auto_mask_substitution" if suffix else "original_baseline"
        predictions.append(table)
        summaries.append({"model": name, **classification_metrics(val.label, prob, threshold)})
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(summaries)







In [ ]:
from __future__ import annotations

def baseline_contract(cfg):
    return definition_contract(cfg, "02", list(ORIGINAL_INFERENCE_CONTRACT))






ORIGINAL_INFERENCE_CONTRACT = {'MaskEncoder': 'b0a35f2673a19f24a5f492a757a65894b572a9819996d2aabb9572cc89b2de04', 'DualBranchModel': '75a158f454de456d8b6e452309c35eb101124771da83567a475d8af2e386bc95', 'predict_proba': '94646b415f42d60d4a16416a05f26afed9656dd0fba15b09323c50f226898663'}


## 恢复优化模型

这组定义与02的优化网络一致，载入时核对权重、参数、队列与缓存。推理直接从已保存权重恢复，不需要下载ImageNet预训练参数。


In [ ]:
from __future__ import annotations

class GatedSliceAttention(nn.Module):
    def __init__(self, features, hidden):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, features):
        return self.score(torch.tanh(self.value(features)) * torch.sigmoid(self.gate(features)))






class CompactMaskEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(.3))

    def forward(self, masks):
        return self.features(masks.float())






class OptimizedResNet(nn.Module):
    def __init__(self, settings, pretrained=True):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_projection = nn.Sequential(nn.Linear(2048, settings["image_dim"]),
            nn.LayerNorm(settings["image_dim"]), nn.GELU(), nn.Dropout(.3))
        self.mask_features = CompactMaskEncoder(settings["mask_dim"])
        features = settings["image_dim"] + settings["mask_dim"]
        self.attention = GatedSliceAttention(features, settings["attention_dim"])
        self.head = nn.Sequential(nn.LayerNorm(features), nn.Dropout(.4), nn.Linear(features, 1))

    def train(self, mode=True):
        super().train(mode)
        # Keep pretrained running means/variances stable with small patient batches.
        for module in self.image_features.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        return self

    def set_training_stage(self, fine_tune):
        for parameter in self.image_features.parameters():
            parameter.requires_grad_(False)
        if fine_tune:
            for parameter in self.image_features[7].parameters():
                parameter.requires_grad_(True)
        for module in self.image_features.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                for parameter in module.parameters():
                    parameter.requires_grad_(False)

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        image = self.image_features(images.reshape(b * slices, channels, h, w)).flatten(1)
        image = self.image_projection(image).reshape(b, slices, -1)
        mask = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
        features = torch.cat([image, mask], dim=-1)
        attention = torch.softmax(self.attention(features).squeeze(-1), dim=1)
        return self.head((features * attention.unsqueeze(-1)).sum(dim=1)).squeeze(-1)






def optimized_contract(cfg):
    return definition_contract(cfg, "02", ["GatedSliceAttention", "CompactMaskEncoder", "OptimizedResNet",
        "optimized_predict", "fit_optimized_epochs", "run_optimized_training", "optimized_data",
        "seed_everything", "augment_batch", "make_criterion"])






def verify_optimized(cfg):
    p = paths(cfg)
    lock = json.loads((p["optimized"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_identity"] != cohort_identity(cfg) or lock["cache_sha256"] != digest(p["cache"]) or lock["code_contract"] != optimized_contract(cfg):
        raise ValueError("优化模型与当前患者、缓存或实现不匹配。")
    for name, checksum in lock["artifacts"].items():
        if digest(p["optimized"] / name) != checksum:
            raise ValueError(f"优化模型文件发生变化：{name}")
    settings = json.loads((p["optimized"] / "training_config.json").read_text(encoding="utf-8"))["settings"]
    if settings != cfg["optimized"]:
        raise ValueError("优化参数与训练时不一致；加载已有模型时使用原参数，调参请更换 experiment。")
    return settings






def load_optimized_model(cfg, checkpoint=None):
    settings = verify_optimized(cfg)
    model = OptimizedResNet(settings, pretrained=False).to(DEVICE)
    checkpoint = checkpoint or paths(cfg)["optimized"] / "final_model.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
    model.eval()
    return model






def optimized_data(cfg, scope):
    table = read_cohort(cfg).query("split == @scope").reset_index(drop=True)
    indices = table.cache_index.to_numpy(dtype=int)
    with np.load(paths(cfg)["cache"], allow_pickle=False) as cache:
        if cache["mrs"][indices].astype(str).tolist() != table.mr.tolist() or not np.array_equal(cache["labels"][indices], table.label):
            raise ValueError("缓存患者或结局与队列不一致。")
        if not np.array_equal(cache["max_layers"][indices], table.max_layer):
            raise ValueError("缓存取层与队列不一致。")
        images = torch.from_numpy(cache["images"][indices].astype(np.float32))
        masks = torch.from_numpy(cache["masks"][indices].astype(np.uint8))
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("优化版继续使用四层、三序列、224×224的原缓存。")
    return table, images, masks, table.label.to_numpy(dtype=int)






def optimized_predict(model, images, masks, batch_size=16):
    model.eval()
    result = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(images[start:start + batch_size].to(DEVICE), masks[start:start + batch_size].to(DEVICE))
            # Store float32 probabilities; avoid rounding at the final sigmoid in half precision.
            result.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(result)






def predict_optimized_validation(cfg):
    table, images, masks, y = optimized_data(cfg, "validation")
    probability = optimized_predict(load_optimized_model(cfg), images, masks, cfg["optimized"]["batch_size"])
    meta = json.loads((paths(cfg)["optimized"] / "development_metrics.json").read_text(encoding="utf-8"))
    result = table[["mr", "label"]].copy()
    result["model"], result["probability"], result["threshold"] = "ResNet50_Optimized", probability, meta["threshold"]
    result["prediction"] = (probability >= meta["threshold"]).astype(int)
    result["protocol"] = "optimized_resnet_inner_epoch_selection"
    return result, pd.DataFrame([{"model": "ResNet50_Optimized", **classification_metrics(y, probability, meta["threshold"])}])






## 用已保存的影像组学模型预测

`predict_radiomics_validation` 核对特征表和模型文件，按 MR 编号将时间验证患者与特征一一匹配，然后加载 01 保存的处理流程和逻辑回归模型。

缺失值填补、特征选择和标准化均使用开发阶段学到的参数，分类阈值也从已保存的记录读取。输出格式与 ResNet 组一致，便于合并比较。


In [ ]:
from __future__ import annotations

def predict_radiomics_validation(cfg):
    p, cohort = paths(cfg), read_cohort(cfg)
    receipt = p["radiomics"] / "transfer_manifest.json"
    if receipt.exists():
        manifest = verify_transfer(cfg, p["radiomics"])
        if manifest["kind"] == "radiomics_complete":
            return imported_radiomics_predictions(cfg)
    import joblib
    lock = json.loads((p["radiomics"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("固定队列发生变化。")
    for name, expected in lock["artifacts"].items():
        if digest(p["radiomics"] / name) != expected:
            raise ValueError(f"影像组学固定产物发生变化：{name}")
    feature_table = pd.read_csv(p["radiomics"] / "features.csv", dtype={"mr": str})
    val = cohort.query("split == 'validation'").merge(feature_table.drop(columns=["label", "split"]), on="mr", validate="one_to_one")
    models = joblib.load(p["radiomics"] / "models.joblib")
    meta = json.loads((p["radiomics"] / "model_meta.json").read_text(encoding="utf-8"))
    predictions, summaries = [], []
    for name, model in models.items():
        prob = model.predict_proba(val[meta[name]["features"]].replace([np.inf, -np.inf], np.nan))[:, 1]
        threshold = meta[name]["threshold"]
        table = val[["mr", "label"]].copy()
        table["model"], table["probability"], table["threshold"] = name, prob, threshold
        table["prediction"] = (prob >= threshold).astype(int)
        table["protocol"] = "radiomics_selected_four_slices_nested_feature_selection"
        predictions.append(table)
        summaries.append({"model": name, **classification_metrics(val.label, prob, threshold)})
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(summaries)






def imported_radiomics_predictions(cfg):
    root = paths(cfg)["radiomics"]
    verify_transfer(cfg, root, "radiomics_complete")
    table = pd.read_csv(root / "validation_predictions.csv", dtype={"mr": str})
    expected = read_cohort(cfg).query("split == 'validation'").set_index("mr")
    meta = json.loads((root / "model_meta.json").read_text(encoding="utf-8"))
    if set(table.model) != {"Radiomics", "Radiomics_Clinical5"}:
        raise ValueError("影像组学预测缺少对照组。")
    summaries = []
    for name, group in table.groupby("model"):
        if not group.mr.is_unique or set(group.mr) != set(expected.index):
            raise ValueError("导入预测的时间验证患者不一致。")
        if not np.array_equal(group.label, expected.loc[group.mr, "label"]):
            raise ValueError("导入预测结局不一致。")
        threshold = float(meta[name]["threshold"])
        if not np.allclose(group.threshold, threshold, rtol=0, atol=1e-12):
            raise ValueError("导入预测阈值与开发阶段不同。")
        if not np.array_equal(group.prediction, (group.probability >= threshold).astype(int)):
            raise ValueError("导入预测分类与概率不一致。")
        summaries.append({"model": name, **classification_metrics(group.label, group.probability, threshold)})
    return table, pd.DataFrame(summaries)







## 评价自动勾画，并转换为分类输入

`evaluate_masks` 在指定四层上逐患者、逐标签比较自动与人工掩模，计算 Dice 和以像素为单位的平面 HD95。没有可比较前景的情况会单独记录状态，汇总时同时显示有效病例数。

`build_validation_mask_cache` 按已有取层记录截取四层，拆分三类病灶，并用最近邻插值缩放到 224 × 224。`verify_validation_mask_cache` 核对转换后的缓存与原生预测文件是否属于同一次结果，确保分割评价和分类实验使用相同的掩模。


In [ ]:
"""Native inputs for the confirmed four annotated slices; no unannotated background."""
from pathlib import Path
import json
import re
import numpy as np


def native_patient_files(cfg, patient):
    require_native_images(cfg)
    columns = SEQUENCES + ["manual_mask"]
    recorded = [Path(str(patient.get(c, ""))) for c in columns]
    if all(p.is_file() for p in recorded):
        return recorded
    root = Path(cfg.get("patients_root") or "")
    if not root.is_dir():
        raise FileNotFoundError("在 config.json 填写数据电脑上的 patients_root。")
    if "_native_lookup" not in cfg:
        lookup = {}
        for file in root.rglob(IMAGE_FILES[0]):
            matches = set(re.findall(r"(?<![A-Z0-9])MR\d+(?!\d)", str(file.parent).upper()))
            for mr in matches:
                lookup.setdefault(mr, []).append(file.parent)
        cfg["_native_lookup"] = lookup
    matches = cfg["_native_lookup"].get(str(patient.mr).upper(), [])
    if len(matches) != 1:
        raise ValueError(f"{patient.mr} 未找到唯一的三序列目录；检查 patients_root 和目录中的 MR 编号。")
    files = [matches[0] / name for name in IMAGE_FILES + ["Untitled.nii.gz"]]
    if not all(file.is_file() for file in files):
        raise FileNotFoundError(f"{patient.mr} 缺少三序列影像或人工标注。")
    return files






def selected_indices(patient, depth):
    value = float(patient.max_layer)
    if not np.isfinite(value) or value != int(value):
        raise ValueError(f"{patient.mr} 起始层不是整数。")
    start = int(value) - 1
    zs = list(range(start, start + 4))
    if start < 0 or zs[-1] >= depth:
        raise ValueError(f"{patient.mr} 的四层取层越界。")
    recorded = patient.get("selected_slices_1based")
    if isinstance(recorded, str) and recorded.strip() and json.loads(recorded) != [z + 1 for z in zs]:
        raise ValueError(f"{patient.mr} 的缓存取层与标注层记录不一致。")
    return zs






def selected_patient(cfg, patient):
    import nibabel as nib
    if cfg.get("annotation_scope") != "selected_slices":
        raise ValueError("本流程要求确认全队列仅选定四层标注。")
    files = native_patient_files(cfg, patient)
    volumes = [nib.load(str(file)) for file in files]
    ref = volumes[0]
    tolerance = np.array([1e-4, 1e-4, 1e-4, 1e-3])
    if len(ref.shape) != 3 or any(v.shape != ref.shape or not np.allclose(v.affine, ref.affine, rtol=0, atol=tolerance) for v in volumes[1:]):
        raise ValueError(f"{patient.mr} 的 shape/affine 不一致，先回到00核查。")
    zs = selected_indices(patient, ref.shape[2])
    truth = np.asarray(volumes[-1].dataobj)
    if not np.isfinite(truth).all() or not set(np.unique(truth)) <= {0, 1, 2, 3}:
        raise ValueError(f"{patient.mr} 人工标签必须为0/1/2/3。")
    outside = np.ones(ref.shape[2], dtype=bool)
    outside[zs] = False
    if np.any(truth[:, :, outside] != 0):
        raise ValueError(f"{patient.mr} 在指定四层之外发现标注，请核对起始层。")
    images = [np.asarray(v.dataobj, dtype=np.float32)[:, :, zs] for v in volumes[:3]]
    if any(not np.isfinite(v).all() for v in images):
        raise ValueError(f"{patient.mr} 影像包含非有限值。")
    checksums = {file.name: digest(file) for file in files}
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    expected = audit["patients"][str(patient.mr)]["files"]
    if any(checksums[name] != expected[name]["sha256"] for name in checksums):
        raise ValueError(f"{patient.mr} 原始影像或标注与00的缓存来源不同，请先核查数据版本。")
    return images, truth[:, :, zs].astype(np.uint8), {
        "mr": str(patient.mr), "slices_1based": [z + 1 for z in zs], "shape": list(ref.shape),
        "native_affine": ref.affine.tolist(), "working_space": "pixel",
        "files_sha256": checksums}





"""Patient-grouped 2D nnU-Net on the four annotated slices, with portable artifacts."""
import json
import os
import shlex
import shutil
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd


def locations(cfg):
    root = paths(cfg)["segmentation"]
    dataset_id = int(cfg["segmentation"]["dataset_id"])
    if not 100 <= dataset_id <= 999 or cfg["segmentation"]["configuration"] != "2d":
        raise ValueError("四层标注使用100～999的数据集ID和2d配置。")
    name = f"Dataset{dataset_id:03d}_PSCISelected2D"
    return {"root": root, "name": name, "raw": root / "nnUNet_raw" / name,
        "preprocessed": root / "nnUNet_preprocessed" / name, "results": root / "nnUNet_results",
        "validation": root / "validation_predictions", "oof": root / "development_oof_predictions"}






def segmentation_label_names(cfg):
    # Numeric placeholders preserve IDs without inventing a clinical interpretation.
    names = {str(i): cfg.get("label_names", {}).get(str(i)) or f"label_{i}" for i in (1, 2, 3)}
    if len(set(names.values()) | {"background"}) != 4:
        raise ValueError("标签名称须唯一且不能命名为background。")
    return names






def export_dataset(cfg):
    import nibabel as nib
    require_native_images(cfg)
    loc, cohort = locations(cfg), read_cohort(cfg)
    fresh(loc["root"])
    names = segmentation_label_names(cfg)
    for directory in ("imagesTr", "labelsTr", "imagesTs"):
        (loc["raw"] / directory).mkdir(parents=True, exist_ok=True)
    rows, sources = [], []
    # Artificial working geometry: one slice, original in-plane resolution, distances in pixels.
    affine = np.diag([1.0, 1.0, 999.0, 1.0])
    for _, patient in cohort.iterrows():
        images, truth, source = selected_patient(cfg, patient)
        sources.append(source)
        training = patient.split == "development"
        folder = loc["raw"] / ("imagesTr" if training else "imagesTs")
        for index, layer in enumerate(source["slices_1based"]):
            case = f"{patient.mr}_s{layer:03d}"
            rows.append({"case_id": case, "mr": patient.mr, "slice_1based": layer,
                         "slice_offset": index, "split": patient.split, "development_fold": int(patient.development_fold)})
            for channel, volume in enumerate(images):
                nib.save(nib.Nifti1Image(volume[:, :, index:index + 1].astype(np.float32), affine), folder / f"{case}_{channel:04d}.nii.gz")
            if training:
                nib.save(nib.Nifti1Image(truth[:, :, index:index + 1], affine), loc["raw"] / "labelsTr" / f"{case}.nii.gz")
    cases = pd.DataFrame(rows)
    cases.to_csv(loc["root"] / "case_index.csv", index=False)
    dev = cases.query("split == 'development'")
    if set(dev.development_fold) != {1, 2, 3, 4, 5}:
        raise ValueError("开发患者必须沿用00中的五折。")
    splits = []
    for fold in range(1, 6):
        valid = dev.loc[dev.development_fold == fold, "case_id"].tolist()
        train = dev.loc[dev.development_fold != fold, "case_id"].tolist()
        splits.append({"train": train, "val": valid})
        inputs = loc["root"] / "oof_inputs" / f"fold_{fold - 1}"
        inputs.mkdir(parents=True)
        for case in valid:
            for channel in range(3):
                name = f"{case}_{channel:04d}.nii.gz"
                shutil.copy2(loc["raw"] / "imagesTr" / name, inputs / name)
    dump_json(loc["raw"] / "dataset.json", {"channel_names": {"0": "T1-FLAIR", "1": "T2", "2": "DWI"},
        "labels": {"background": 0, **{name: int(label) for label, name in names.items()}},
        "numTraining": len(dev), "file_ending": ".nii.gz"})
    dump_json(loc["root"] / "splits_final.json", splits)
    tracked = [file for directory in (loc["raw"], loc["root"] / "oof_inputs") for file in directory.rglob("*") if file.is_file()]
    tracked += [loc["root"] / "case_index.csv", loc["root"] / "splits_final.json"]
    dump_json(loc["root"] / "dataset_provenance.json", {"cohort_identity": cohort_identity(cfg),
        "validation_labels_in_training": False, "annotation_scope": "selected_slices", "configuration": "2d",
        "protocol": "selected_four_slices_2d_pixels_v1", "label_names": names, "sources": sources,
        "artifacts": {file.relative_to(loc["root"]).as_posix(): digest(file) for file in tracked}})
    record(cfg, "segmentation_dataset", {"protocol": "selected_four_slices_2d_pixels_v1"})
    return loc






def verify_dataset(cfg):
    loc, cohort = locations(cfg), read_cohort(cfg)
    meta = json.loads((loc["root"] / "dataset_provenance.json").read_text(encoding="utf-8"))
    if meta["cohort_identity"] != cohort_identity(cfg) or meta["annotation_scope"] != cfg["annotation_scope"]:
        raise ValueError("分割导出后患者、划分、取层或标注协议发生变化。")
    if meta["configuration"] != cfg["segmentation"]["configuration"] or meta["label_names"] != segmentation_label_names(cfg):
        raise ValueError("分割配置或标签定义发生变化，请更换 segmentation.experiment。")
    for name, checksum in meta["artifacts"].items():
        if digest(loc["root"] / name) != checksum:
            raise ValueError(f"分割数据文件变化：{name}")
    dev = cohort.query("split == 'development'")
    expected = {f"{patient.mr}_s{layer:03d}" for _, patient in dev.iterrows()
                for layer in range(int(patient.max_layer), int(patient.max_layer) + 4)}
    actual = {file.name.removesuffix(".nii.gz") for file in (loc["raw"] / "labelsTr").glob("*.nii.gz")}
    if actual != expected:
        raise ValueError("labelsTr与开发患者四层清单不一致。")






def command_plan(cfg):
    loc, settings = locations(cfg), cfg["segmentation"]
    dataset = str(settings["dataset_id"])
    plan = {"preprocess": [["nnUNetv2_plan_and_preprocess", "-d", dataset, "-c", "2d", "--verify_dataset_integrity"]],
            "train": [], "predict_oof": [], "predict_validation": []}
    for fold in range(5):
        plan["train"].append(["nnUNetv2_train", dataset, "2d", str(fold), "--npz", "-device", settings["device"]])
        plan["predict_oof"].append(["nnUNetv2_predict", "-i", str(loc["root"] / "oof_inputs" / f"fold_{fold}"),
            "-o", str(loc["oof"] / f"fold_{fold}"), "-d", dataset, "-c", "2d", "-f", str(fold), "-device", settings["device"]])
    plan["predict_validation"] = [["nnUNetv2_predict", "-i", str(loc["raw"] / "imagesTs"),
        "-o", str(loc["validation"]), "-d", dataset, "-c", "2d", "-f", "0", "1", "2", "3", "4", "-device", settings["device"]]]
    return plan






def install_splits(cfg):
    verify_dataset(cfg)
    loc = locations(cfg)
    if not loc["preprocessed"].is_dir():
        raise FileNotFoundError("先完成预处理，再安装固定的患者划分。")
    source, target = loc["root"] / "splits_final.json", loc["preprocessed"] / "splits_final.json"
    if target.exists():
        if json.loads(target.read_text()) != json.loads(source.read_text()):
            raise ValueError("已有nnU-Net划分与固定患者折不同，不覆盖。")
    else:
        shutil.copy2(source, target)
    return target






def run_phase(cfg, phase):
    verify_dataset(cfg)
    loc = locations(cfg)
    if phase not in command_plan(cfg):
        raise ValueError("未知分割阶段。")
    if phase != "preprocess":
        install_splits(cfg)
    env = os.environ.copy()
    env.update(nnUNet_raw=str(loc["raw"].parent), nnUNet_preprocessed=str(loc["preprocessed"].parent), nnUNet_results=str(loc["results"]))
    for folder in (loc["preprocessed"].parent, loc["results"]):
        folder.mkdir(parents=True, exist_ok=True)
    for command in command_plan(cfg)[phase]:
        print(shlex.join(command), flush=True)
        subprocess.run(command, check=True, env=env)
    record(cfg, "segmentation_" + phase, {"protocol": "selected_four_slices_2d_pixels_v1"})






def predicted_slices(cfg, patient, scope, shape):
    import nibabel as nib
    loc = locations(cfg)
    directory = loc["validation"] if scope == "validation" else loc["oof"] / f"fold_{int(patient.development_fold) - 1}"
    arrays, checksums = [], {}
    for layer in range(int(patient.max_layer), int(patient.max_layer) + 4):
        name = f"{patient.mr}_s{layer:03d}.nii.gz"
        volume = nib.load(str(directory / name))
        values = np.asarray(volume.dataobj)
        if values.shape != (*shape, 1) or not np.allclose(volume.affine, np.diag([1., 1., 999., 1.]), rtol=0, atol=1e-4):
            raise ValueError(f"{name} 预测工作网格不一致。")
        if not np.isfinite(values).all() or not set(np.unique(values)) <= {0, 1, 2, 3}:
            raise ValueError(f"{name} 预测标签不是0/1/2/3。")
        arrays.append(values[:, :, 0].astype(np.uint8))
        checksums[name] = digest(directory / name)
    return np.stack(arrays, axis=2), checksums






def evaluate_masks(cfg, scope="validation"):
    from scipy.ndimage import binary_erosion, distance_transform_edt
    require_native_images(cfg)
    verify_dataset(cfg)
    if scope not in ("validation", "development_oof"):
        raise ValueError("scope只能是validation或development_oof。")
    loc, cohort = locations(cfg), read_cohort(cfg)
    cases = cohort.query("split == 'validation'" if scope == "validation" else "split == 'development'")
    rows, prediction_hashes = [], {}
    for _, patient in cases.iterrows():
        _, truth, _ = selected_patient(cfg, patient)
        prediction, hashes = predicted_slices(cfg, patient, scope, truth.shape[:2])
        prediction_hashes.update(hashes)
        for label in (1, 2, 3):
            a, b = truth == label, prediction == label
            na, nb = int(a.sum()), int(b.sum())
            distances, undefined = [], False
            for index in range(4):
                ai, bi = a[:, :, index], b[:, :, index]
                if not ai.any() and not bi.any():
                    continue
                if not ai.any() or not bi.any():
                    undefined = True
                    continue
                sa, sb = ai ^ binary_erosion(ai, border_value=0), bi ^ binary_erosion(bi, border_value=0)
                distances.extend([distance_transform_edt(~sb)[sa], distance_transform_edt(~sa)[sb]])
            rows.append({"mr": patient.mr, "scope": scope, "label": label, "true_pixels": na, "predicted_pixels": nb,
                "dice": float(2 * (a & b).sum() / (na + nb)) if na + nb else np.nan,
                "hd95_pixels": float(np.percentile(np.concatenate(distances), 95)) if distances and not undefined else np.nan,
                "status": "both_empty" if not na + nb else "empty_slice_hd95_undefined" if undefined else "ok"})
    table = pd.DataFrame(rows)
    table.to_csv(loc["root"] / f"{scope}_segmentation_metrics.csv", index=False)
    summary = table.groupby("label").agg(n_cases=("mr", "size"), n_dice_defined=("dice", "count"), mean_dice=("dice", "mean"),
        n_hd95_defined=("hd95_pixels", "count"), mean_hd95_pixels=("hd95_pixels", "mean"))
    summary.to_csv(loc["root"] / f"{scope}_segmentation_summary.csv")
    dump_json(loc["root"] / f"{scope}_evaluation_provenance.json", {
        "cohort_identity": cohort_identity(cfg), "scope": scope, "predictions_sha256": prediction_hashes,
        "artifacts": {name: digest(loc["root"] / name) for name in
            (f"{scope}_segmentation_metrics.csv", f"{scope}_segmentation_summary.csv")}})
    return table, summary






def build_validation_mask_cache(cfg):
    from scipy.ndimage import zoom
    require_native_images(cfg)
    verify_dataset(cfg)
    loc = locations(cfg)
    target = loc["root"] / "validation_masks.npz"
    fresh(target)
    cases = read_cohort(cfg).query("split == 'validation'").reset_index(drop=True)
    result = np.zeros((len(cases), 4, 3, 224, 224), dtype=np.uint8)
    hashes = {}
    for i, patient in cases.iterrows():
        _, truth, _ = selected_patient(cfg, patient)
        labels, checksums = predicted_slices(cfg, patient, "validation", truth.shape[:2])
        hashes.update(checksums)
        for index in range(4):
            for label in (1, 2, 3):
                result[i, index, label - 1] = zoom((labels[:, :, index] == label).astype(np.uint8),
                    (224 / labels.shape[0], 224 / labels.shape[1]), order=0, prefilter=False)
    np.savez_compressed(target, mrs=cases.mr.to_numpy(dtype=str), masks=result)
    dump_json(loc["root"] / "automatic_input_note.json", {"cohort_identity": cohort_identity(cfg),
        "scope": "validation_only", "experiment": "manual_model_auto_mask_substitution",
        "manual_slice_selection_still_required": True, "protocol": "selected_four_slices_2d_pixels_v1",
        "mask_cache_sha256": digest(target), "predictions_sha256": hashes,
        "note": "只替换已训练分类器的病灶输入；不代表自动取层或自动掩模训练的联合模型。"})
    return target






def verify_validation_mask_cache(cfg):
    loc = locations(cfg)
    if (loc["root"] / "transfer_manifest.json").exists():
        verify_transfer(cfg, loc["root"], "segmentation_results")
    metadata = json.loads((loc["root"] / "automatic_input_note.json").read_text(encoding="utf-8"))
    if metadata["cohort_identity"] != cohort_identity(cfg) or metadata["protocol"] != "selected_four_slices_2d_pixels_v1":
        raise ValueError("自动掩模与当前患者或取层协议不一致。")
    if digest(loc["root"] / "validation_masks.npz") != metadata["mask_cache_sha256"]:
        raise ValueError("自动分类掩模缓存发生变化。")
    cases = read_cohort(cfg).query("split == 'validation'")
    with np.load(loc["root"] / "validation_masks.npz", allow_pickle=False) as cache:
        if cache["mrs"].astype(str).tolist() != cases.mr.tolist() or cache["masks"].shape != (len(cases), 4, 3, 224, 224):
            raise ValueError("自动掩模患者顺序或形状不一致。")
        if not set(np.unique(cache["masks"])) <= {0, 1} or np.any(cache["masks"].sum(axis=2) > 1):
            raise ValueError("自动掩模必须为互斥的三通道二值标签。")
    if not (loc["root"] / "transfer_manifest.json").exists():
        for name, checksum in metadata["predictions_sha256"].items():
            if digest(loc["validation"] / name) != checksum:
                raise ValueError(f"原预测发生变化：{name}")






def export_segmentation_results(cfg, destination):
    verify_validation_mask_cache(cfg)
    for scope in ("validation", "development_oof"):
        verify_segmentation_evaluation(cfg, scope)
    files = ["validation_masks.npz", "automatic_input_note.json", "dataset_provenance.json", "case_index.csv", "splits_final.json",
        "validation_segmentation_metrics.csv", "validation_segmentation_summary.csv",
        "development_oof_segmentation_metrics.csv", "development_oof_segmentation_summary.csv",
        "validation_evaluation_provenance.json", "development_oof_evaluation_provenance.json"]
    return export_transfer(cfg, locations(cfg)["root"], files, destination, "segmentation_results", "selected_four_slices_2d_pixels_v1")






def export_segmentation_weights(cfg, destination):
    from importlib.metadata import version
    loc = locations(cfg)
    verify_dataset(cfg)
    model = loc["results"] / loc["name"] / "nnUNetTrainer__nnUNetPlans__2d"
    files = [model / "dataset.json", model / "plans.json"]
    if (model / "dataset_fingerprint.json").is_file():
        files.append(model / "dataset_fingerprint.json")
    for fold in range(5):
        folder = model / f"fold_{fold}"
        candidates = [folder / name for name in ("checkpoint_final.pth", "checkpoint_final.safetensors") if (folder / name).is_file()]
        if len(candidates) != 1:
            raise ValueError(f"fold_{fold} 需要唯一的最终权重，当前找到{len(candidates)}个。")
        files += candidates
        files += [file for file in folder.iterdir() if file.suffix in (".json", ".pkl")]
    environment = loc["root"] / "model_environment.json"
    dump_json(environment, {"nnunetv2": version("nnunetv2"), "torch": version("torch"),
        "model_relative_path": model.relative_to(loc["root"]).as_posix(),
        "input": "native-resolution selected slices exported by this notebook; not the 224x224 classifier cache"})
    files += [environment, loc["root"] / "dataset_provenance.json", loc["root"] / "splits_final.json"]
    return export_transfer(cfg, loc["root"], [file.relative_to(loc["root"]).as_posix() for file in files],
                           destination, "segmentation_weights", "selected_four_slices_2d_pixels_v1")






def imported_segmentation_inference_plan(cfg, input_folder, output_folder):
    root = paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    verify_transfer(cfg, root, "segmentation_weights")
    meta = json.loads((root / "model_environment.json").read_text(encoding="utf-8"))
    from importlib.metadata import version
    if version("nnunetv2") != meta["nnunetv2"]:
        raise ValueError("恢复推理时请使用导出记录中的 nnunetv2 版本：" + meta["nnunetv2"])
    return ["nnUNetv2_predict_from_modelfolder", "-i", str(input_folder), "-o", str(output_folder),
            "-m", str(root / meta["model_relative_path"]), "-f", "0", "1", "2", "3", "4", "-device", cfg["segmentation"]["device"]]






def verify_segmentation_evaluation(cfg, scope):
    loc = locations(cfg)
    meta = json.loads((loc["root"] / f"{scope}_evaluation_provenance.json").read_text(encoding="utf-8"))
    if meta["cohort_identity"] != cohort_identity(cfg) or meta["scope"] != scope:
        raise ValueError("分割评价与当前患者或评价范围不同。")
    for name, checksum in meta["artifacts"].items():
        if digest(loc["root"] / name) != checksum:
            raise ValueError(f"分割指标发生变化：{name}")
    cases = read_cohort(cfg).set_index("mr")
    for name, checksum in meta["predictions_sha256"].items():
        mr = name.split("_s", 1)[0]
        directory = loc["validation"] if scope == "validation" else loc["oof"] / f"fold_{int(cases.loc[mr, 'development_fold']) - 1}"
        if digest(directory / name) != checksum:
            raise ValueError(f"{name}在评价后发生变化，请重新评价。")




## 统一指标、模型比较与绘图

`classification_metrics` 根据每组已确定的阈值计算准确率、敏感度、特异度和混淆矩阵，同时报告 AUC、AUC 重采样区间及 Brier 分数。

`paired_comparison` 按患者编号对齐两个模型，在同一批抽样患者上计算 AUC 差值及区间。`plot_predictions` 绘制 ROC 和校准图：前者观察区分能力，后者把预测概率与实际阳性比例放在一起比较。

`run_validation_report` 将各组预测合并，分别与临床模型比较，并在纳入影像组学时增加 ResNet 与影像组学的配对比较，最后保存指标表、患者预测表和图片。


In [ ]:
"""Classification summaries; thresholds are supplied, never selected on validation."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils import resample


def threshold_from_development(y, probability):
    fpr, tpr, thresholds = roc_curve(y, probability)
    return float(thresholds[np.argmax(tpr - fpr)])







def classification_metrics(y, probability, threshold, n_bootstrap=1000):
    y, probability = np.asarray(y, dtype=int), np.asarray(probability, dtype=float)
    if len(y) != len(probability) or set(y) != {0, 1} or not np.isfinite(probability).all():
        raise ValueError("标签需包含两类，概率必须有限且与患者一一对应。")
    if ((probability < 0) | (probability > 1)).any():
        raise ValueError("概率不在[0,1]。")
    tn, fp, fn, tp = confusion_matrix(y, probability >= threshold, labels=[0, 1]).ravel()
    result = {"n": len(y), "auc": float(roc_auc_score(y, probability)), "threshold": float(threshold),
              "accuracy": float((tp + tn) / len(y)), "sensitivity": float(tp / (tp + fn)),
              "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(y, probability)),
              "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}
    if n_bootstrap:
        values = []
        for seed in range(n_bootstrap):
            idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
            values.append(roc_auc_score(y[idx], probability[idx]))
        result.update(auc_ci_low=float(np.percentile(values, 2.5)), auc_ci_high=float(np.percentile(values, 97.5)))
    return result







def paired_comparison(a: pd.DataFrame, b: pd.DataFrame, n_bootstrap=1000):
    for frame in (a, b):
        if not frame["mr"].is_unique or set(frame["label"]) != {0, 1}:
            raise ValueError("比较需要唯一患者编号与两类结局。")
    if set(a["mr"]) != set(b["mr"]):
        raise ValueError("两组患者不完全一致；不静默取交集。")
    merged = a.merge(b, on="mr", suffixes=("_a", "_b"), validate="one_to_one")
    if not (merged.label_a == merged.label_b).all():
        raise ValueError("同一患者的标签不一致。")
    y, pa, pb = merged.label_a.to_numpy(), merged.probability_a.to_numpy(), merged.probability_b.to_numpy()
    if not np.isfinite(pa).all() or not np.isfinite(pb).all():
        raise ValueError("预测包含非有限值。")
    differences = []
    for seed in range(n_bootstrap):
        idx = resample(np.arange(len(y)), replace=True, stratify=y, random_state=seed)
        differences.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.asarray(differences)
    return {"auc_difference": float(roc_auc_score(y, pa) - roc_auc_score(y, pb)),
            "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5)),
            "empirical_p": min(1.0, 2 * min(float((d <= 0).mean()), float((d >= 0).mean())))}







def plot_predictions(predictions):
    import matplotlib.pyplot as plt
    from sklearn.calibration import calibration_curve
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, table in predictions.groupby("model"):
        fpr, tpr, _ = roc_curve(table.label, table.probability)
        axes[0].plot(fpr, tpr, label=f"{name}: {roc_auc_score(table.label, table.probability):.3f}")
        truth, prob = calibration_curve(table.label, table.probability, n_bins=5, strategy="quantile")
        axes[1].plot(prob, truth, "o-", label=name)
    for ax in axes:
        ax.plot([0, 1], [0, 1], "k--", alpha=.4)
        ax.legend(fontsize=8)
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    axes[1].set(xlabel="Predicted probability", ylabel="Observed proportion", title="Calibration (no refitting)")
    fig.tight_layout()
    return fig







def run_validation_report(cfg, include_radiomics=True, include_automatic=False, include_optimized=False):
    p = paths(cfg)
    fresh(p["reports"] / "validation_predictions.csv", p["reports"] / "validation_metrics.csv")
    predictions, metrics = predict_original_validation(cfg)
    tables, summaries = [predictions], [metrics]
    if include_radiomics:
        predicted, summary = predict_radiomics_validation(cfg)
        tables.append(predicted)
        summaries.append(summary)
    if include_automatic:
        automatic_cache = locations(cfg)["root"] / "validation_masks.npz"
        if not automatic_cache.exists():
            build_validation_mask_cache(cfg)
        verify_validation_mask_cache(cfg)
        predicted, summary = predict_original_validation(cfg, automatic_masks=automatic_cache)
        tables.append(predicted)
        summaries.append(summary)
        if not (locations(cfg)["root"] / "transfer_manifest.json").exists():
            evaluate_masks(cfg, scope="validation")
    if include_optimized:
        predicted, summary = predict_optimized_validation(cfg)
        tables.append(predicted)
        summaries.append(summary)
    all_predictions = pd.concat(tables, ignore_index=True)
    all_metrics = pd.concat(summaries, ignore_index=True)
    p["reports"].mkdir(parents=True, exist_ok=True)
    all_predictions.to_csv(p["reports"] / "validation_predictions.csv", index=False)
    all_metrics.to_csv(p["reports"] / "validation_metrics.csv", index=False)
    groups = {name: table for name, table in all_predictions.groupby("model")}
    comparisons = []
    for name in groups:
        if name != "Clinical5":
            comparisons.append({"a": name, "b": "Clinical5", **paired_comparison(groups[name], groups["Clinical5"])})
    if "Radiomics" in groups:
        comparisons.append({"a": "ResNet50", "b": "Radiomics", **paired_comparison(groups["ResNet50"], groups["Radiomics"])})
    if "ResNet50_Optimized" in groups:
        comparisons.append({"a": "ResNet50_Optimized", "b": "ResNet50", **paired_comparison(groups["ResNet50_Optimized"], groups["ResNet50"])})
    pd.DataFrame(comparisons).to_csv(p["reports"] / "paired_auc_comparisons.csv", index=False)
    figure = plot_predictions(all_predictions)
    figure.savefig(p["reports"] / "roc_calibration.png", dpi=150)
    import matplotlib.pyplot as plt
    plt.close(figure)
    dump_json(p["reports"] / "report_notes.json", {
        "resnet": "原结构、训练、单模型及一次OOF晚期融合；没有新增校准或全流程嵌套改造。",
        "radiomics": "四层逐层二维特征后按患者聚合；空间单位为像素。" if include_radiomics else "未纳入本次报告。",
        "automatic": "人工掩模训练的原分类器替换输入敏感性实验；仍需人工取层。" if include_automatic else "未纳入本次报告。",
        "optimized": "独立内层选轮数、外层重新训练的ResNet候选；不包含新的临床融合。" if include_optimized else "未纳入本次报告。",
        "auc_p": "经验配对bootstrap；不是DeLong检验。"})
    record(cfg, "validation_report")
    return all_metrics, pd.DataFrame(comparisons)







## 1. 查看模型和输入是否齐备

检查患者清单、分类模型校验清单、影像组学模型清单和可选自动掩模目录是否存在。这里先查看文件准备情况，正式预测时还会逐项核对内容和版本。


In [ ]:
import pandas as pd
readiness = {
    "固定病例": p["cohort"],
    "原ResNet及临床融合锁定清单": p["temporal"] / "frozen_artifacts.json",
    "影像组学锁定清单": p["radiomics"] / "frozen_artifacts.json",
    "自动掩模目录（可选）": p["segmentation"] / "validation_predictions",
}
display(pd.DataFrame([{"item": key, "ready": value.exists(), "path": str(value)} for key, value in readiness.items()]))


## 2. 运行时间验证

原基线三组默认保留。优化模型完成后打开 `INCLUDE_OPTIMIZED`；01导入完整影像组学结果包或完成本机训练后打开 `INCLUDE_RADIOMICS`；05导入分割结果包后打开 `INCLUDE_AUTOMATIC_MASK_SUBSTITUTION`。只有分类缓存的算力机也能比较导入结果。

结果写到 `reports/comparison_v2/`，保留之前的报告。需要新增比较时更换 `experiment_settings.json` 的 `report_version`；不要依据时间验证结果反复选择参数。


In [ ]:
RUN_VALIDATION = False
INCLUDE_RADIOMICS = False
INCLUDE_AUTOMATIC_MASK_SUBSTITUTION = False
INCLUDE_OPTIMIZED = False

if RUN_VALIDATION:
    metrics, comparisons = run_validation_report(cfg, include_radiomics=INCLUDE_RADIOMICS, include_automatic=INCLUDE_AUTOMATIC_MASK_SUBSTITUTION, include_optimized=INCLUDE_OPTIMIZED)
    display(metrics)
    display(comparisons)


## 3. 查看总体指标与曲线

`validation_metrics.csv` 给出各模型的指标，`paired_auc_comparisons.csv` 给出模型间 AUC 差值和区间。差值为表中 A 模型减 B 模型，可结合区间判断差异估计的精确程度。

ROC 图展示各模型的区分表现；校准图将患者按预测概率分组，比较平均预测概率与实际阳性比例。校准图用于观察概率是否偏高或偏低，此处不会据此重新拟合模型。


In [ ]:
from IPython.display import Image, display
for filename in ["validation_metrics.csv", "paired_auc_comparisons.csv"]:
    target = p["reports"] / filename
    if target.exists():
        print(filename)
        display(pd.read_csv(target))
figure = p["reports"] / "roc_calibration.png"
if figure.exists():
    display(Image(filename=str(figure)))


## 4. 回到患者层面查看误判

读取每位患者的真实标签、预测概率、阈值和最终分类，按模型汇总误判数量，再列出具体病例。可以结合 04 的病例图和临床资料，分析错误是否集中在某些患者特征上。

AUC 描述概率对两类患者的排序能力；准确率、敏感度和特异度描述当前阈值下的分类结果，报告时需要一起看。


In [ ]:
target = p["reports"] / "validation_predictions.csv"
if target.exists():
    predictions = pd.read_csv(target)
    display(predictions.head())
    errors = predictions.loc[predictions.prediction != predictions.label]
    display(errors.groupby("model").size().rename("n_errors"))
    display(errors[["model", "mr", "label", "probability", "threshold", "prediction"]])


## 整理研究结果时需要交代的条件

时间验证结论取决于实际纳入患者、模型版本和本次生成的结果文件。这批验证队列原计划为 61 例，规模较小，既往结果也已被查看；后续若据此反复挑选模型，评价会逐渐偏向这批患者。

方法部分还应说明：ResNet 的开发折参与过最佳轮数选择，融合使用一次生成的影像折外概率；影像组学和ResNet均限定四层，但前者使用原始平面分辨率和二维汇总；优化ResNet只在内层选轮数；自动掩模组使用的是人工掩模训练的分类器。

AUC 区间和配对比较基于固定预测的患者重采样，反映病例抽样的不确定性，不包含重新训练模型带来的全部波动。表中的经验 p 值来自配对 bootstrap，统计方法应按实际计算过程注明。


异地模型校验围绕患者、取层、划分和数据文件，不要求电脑盘符相同。原分类基线仍检查权重、临床模型、阈值及缓存；其网络和推理实现也要与训练版本一致。Notebook配文或其他任务的修改不会使完整数据包失效。
